# 06 — COMET-QN: Quantile Renormalisation (Tables 6 and 7)

Notebook 05 showed that per-language correction cannot restore ranking. COMET-QN
takes the opposite tack: it does not try to fix any language's internal ordering.
It maps every (language, script) cell onto one **common reference distribution**,
the pooled native-COMET distribution, so that scores from different cells become
mutually comparable.

For a cell of scores `s` and sorted reference `R`:

    QN(s)_i = quantile(R, rank(s_i) / (n + 1))

Two consequences follow directly, and both are checked below:

1. The mean per-language native–romanised gap goes to zero by construction.
2. Within-language Spearman ρ is **unchanged**, because QN is monotone — the same
   invariance that made correction futile in notebook 05 makes renormalisation safe here.

What improves is *pooled* correlation: once cells are on a common scale, scores
from different languages can be compared, and pooled ρ rises.

**Base:** 6,995 (ten (language, script) cells).

**Input:** `../data/indic/indic_parity_xlmr.xlsx`
**Output:** `../results/tables/table7_comet_qn.csv`

## Step 0 — Configuration

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# ── Paths (relative; nothing in this repository uses an absolute path) ───────
DATA_XLMR   = Path("../data/indic/indic_parity_xlmr.xlsx")
DATA_MULTI  = Path("../data/indic/indic_parity_multi_tokenizer.xlsx")
DATA_LATIN  = Path("../data/latin/wmt24_ende_enes_metrics.xlsx")
TABLES_DIR  = Path("../results/tables")
FIGURES_DIR = Path("../results/figures")
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Sheet names exactly as they appear in the workbook ───────────────────────
SHEET_MAP = {
    "GUJ": "Indic_mt _for_analysis - Gujara",
    "TAM": "Indic_mt _for_analysis - Tamil_",
    "MAL": "Indic_mt _for_analysis - Malaya",
    "MAR": "Indic_mt _for_analysis - Marath",
    "HIN": "Indic_mt _for_analysis - Hindi_",
}

# ── Language display order (fixed throughout the paper) ──────────────────────
LANG_ORDER = ["GUJ", "TAM", "MAL", "MAR", "HIN"]

# ── Column names (as in the xlsx) ────────────────────────────────────────────
COL_COMET_NAT = "COMET"                                          # native-script COMET
COL_COMET_ROM = "COMET_romanized"                                # romanised COMET
COL_TP_NAT    = "Translation_xlmr_TP"                            # TP, native
COL_TP_ROM    = "Translation_Transliteration_romanized_xlmr_TP"  # TP, romanised
COL_IP_NAT    = "Translation_xlmr_IP"                            # IP, native
COL_IP_ROM    = "Translation_Transliteration_romanized_xlmr_IP"  # IP, romanised
COL_HUMAN     = "Human_scores"                                   # MQM-derived human score
COL_SEVERITY  = "Error1_Severity"                                # primary error severity

# ── Seeds (every stochastic step in this repository) ─────────────────────────
SEED_SPLIT = 42   # 50/50 within-language train/test split
SEED_GBM   = 0    # GradientBoostingRegressor
SEED_PERM  = 0    # paired permutation test

print("Config loaded. DATA_XLMR:", DATA_XLMR)

## Step 1 — Load and Assemble the Working Set

In [ ]:
def load_sheets(path):
    """Read the five per-language sheets.

    Returns two dicts keyed by ISO code:
      full  — all 1,400 rows per language (the 7,000-segment base)
      work  — rows carrying a numeric human score (the 6,995-segment base)

    Coercing the human-score column to numeric is what removes the five
    unusable rows: four are blank and one (Malayalam) holds the string
    ``\`19``, which is not a score.
    """
    full, work = {}, {}
    for lang in LANG_ORDER:
        d = pd.read_excel(path, sheet_name=SHEET_MAP[lang])
        d["H"] = pd.to_numeric(d[COL_HUMAN], errors="coerce")
        full[lang] = d
        work[lang] = d.dropna(subset=["H"]).reset_index(drop=True)
    return full, work


full, work = load_sheets(DATA_XLMR)
print(f"Loaded {sum(len(full[l]) for l in LANG_ORDER):,} rows "
      f"across {len(SHEET_MAP)} sheets")

from scipy import stats

A = pd.concat([work[l].assign(lang=l) for l in LANG_ORDER], ignore_index=True)
A["Cn"]  = A[COL_COMET_NAT]
A["Cr"]  = A[COL_COMET_ROM]
A["TPn"] = A[COL_TP_NAT]
A["TPr"] = A[COL_TP_ROM]
A["IPn"] = A[COL_IP_NAT]
A["IPr"] = A[COL_IP_ROM]
A["dTP"] = A["TPn"] - A["TPr"]      # native minus romanised parity delta
A["dIP"] = A["IPn"] - A["IPr"]
print(f"Working set assembled: N = {len(A):,}")

## Step 2 — The Renormalisation Map

In [ ]:
def qn_map(scores, reference_sorted):
    """Rank -> fractional rank -> empirical quantile of the reference."""
    frac = stats.rankdata(scores) / (len(scores) + 1)
    return np.quantile(reference_sorted, frac)


REFERENCE = np.sort(A["Cn"].values)   # pooled native COMET, the common reference
print(f"Reference distribution: pooled native COMET, n = {len(REFERENCE):,}")
print(f"  min {REFERENCE.min():.2f}   median {np.median(REFERENCE):.2f}   "
      f"max {REFERENCE.max():.2f}")

## Step 3 — Apply COMET-QN to All Ten Cells

In [ ]:
Q = pd.concat([
    pd.DataFrame({
        "lang":  lang,
        "H":     A[A.lang == lang]["H"].values,
        "raw_n": A[A.lang == lang]["Cn"].values,
        "raw_r": A[A.lang == lang]["Cr"].values,
        "qn_n":  qn_map(A[A.lang == lang]["Cn"].values, REFERENCE),
        "qn_r":  qn_map(A[A.lang == lang]["Cr"].values, REFERENCE),
    })
    for lang in LANG_ORDER
], ignore_index=True)

print("Per-language native \u2212 romanised gap, before and after COMET-QN")
print(f"{'Lang':>5}  {'gap before':>11}  {'gap after':>10}")
print("-" * 30)
for lang in LANG_ORDER:
    s = Q[Q.lang == lang]
    print(f"{lang:>5}  {s['raw_n'].mean() - s['raw_r'].mean():>+11.2f}  "
          f"{s['qn_n'].mean() - s['qn_r'].mean():>+10.2f}")

## Step 5 — Table 7

In [ ]:
gap_before = np.mean([abs(Q[Q.lang == l]["raw_n"].mean() - Q[Q.lang == l]["raw_r"].mean())
                      for l in LANG_ORDER])
gap_after = np.mean([abs(Q[Q.lang == l]["qn_n"].mean() - Q[Q.lang == l]["qn_r"].mean())
                     for l in LANG_ORDER])

raw = np.concatenate([Q["raw_n"], Q["raw_r"]])
qn = np.concatenate([Q["qn_n"], Q["qn_r"]])
HH = np.concatenate([Q["H"], Q["H"]])

summary = dict(
    n_cells=10,
    N=len(A),
    gap_before=gap_before,
    gap_after=gap_after,
    pearson_before=stats.pearsonr(raw, HH)[0],
    pearson_after=stats.pearsonr(qn, HH)[0],
    spearman_before=stats.spearmanr(raw, HH)[0],
    spearman_after=stats.spearmanr(qn, HH)[0],
)

print("Table 7 — COMET-QN over the ten (language, script) cells")
print(f"  Mean |per-language gap| (pts) : {gap_before:>6.2f}  \u2192  {gap_after:>6.2f}")
print(f"  Pooled Pearson  (score, H)    : {summary['pearson_before']:>6.3f}  \u2192  "
      f"{summary['pearson_after']:>6.3f}")
print(f"  Pooled Spearman (score, H)    : {summary['spearman_before']:>6.3f}  \u2192  "
      f"{summary['spearman_after']:>6.3f}")

## Step 6 — Rank Invariance

The load-bearing property. If QN altered within-language ranking it would be
just another corrector, and notebook 05 would apply to it. It does not.

In [ ]:
print("Within-language Spearman(native, human), before and after COMET-QN")
print(f"{'Lang':>5}  {'before':>8}  {'after':>8}  {'\u0394':>8}")
print("-" * 34)
within = {}
for lang in LANG_ORDER:
    s = Q[Q.lang == lang]
    b = stats.spearmanr(s["raw_n"], s["H"])[0]
    a = stats.spearmanr(s["qn_n"], s["H"])[0]
    within[lang] = (b, a)
    print(f"{lang:>5}  {b:>8.4f}  {a:>8.4f}  {a - b:>+8.6f}")

# ── Cross-verification against the paper ─────────────────────────────────────
for lang, (b, a) in within.items():
    assert abs(a - b) < 1e-9, f"{lang}: QN changed within-language \u03c1"
assert gap_after < 0.01, f"gap after QN = {gap_after:.4f}, expected \u2248 0"
assert abs(gap_before - 8.14) < 0.01
assert summary["spearman_after"] > summary["spearman_before"]
assert summary["pearson_after"] > summary["pearson_before"]

print(f"\n\u2713 Within-language \u03c1 unchanged to 1e-9 for all five languages (rank-invariant)")
print(f"\u2713 Mean |gap| {gap_before:.2f} \u2192 {gap_after:.2f} pts (paper: 8.14 \u2192 0.00)")
print(f"\u2713 Pooled Spearman {summary['spearman_before']:.3f} \u2192 "
      f"{summary['spearman_after']:.3f} (cross-language comparability improves)")
print(f"\u2713 Pooled Pearson  {summary['pearson_before']:.3f} \u2192 "
      f"{summary['pearson_after']:.3f}")

## Step 7 — Pooled Diagnostic Slopes

How far COMET moves per unit of TP and of IP, pooled over both script conditions.
IP is by far the stronger lever, consistent with the Entropy Penalty dominating
the Computational Tax in notebook 03.

In [ ]:
X_tp = np.concatenate([A["TPn"], A["TPr"]])
X_ip = np.concatenate([A["IPn"], A["IPr"]])
Y = np.concatenate([A["Cn"], A["Cr"]])
slope_tp = np.polyfit(X_tp, Y, 1)[0]
slope_ip = np.polyfit(X_ip, Y, 1)[0]

summary["slope_comet_tp"] = slope_tp
summary["slope_comet_ip"] = slope_ip
print(f"  slope COMET~TP = {slope_tp:6.3f} COMET points per unit TP")
print(f"  slope COMET~IP = {slope_ip:6.3f} COMET points per unit IP")
print(f"\n  IP moves COMET {slope_ip / slope_tp:.1f}\u00d7 more per unit than TP does.")

## Step 7 — Save

In [ ]:
out = pd.DataFrame([summary])
for lang, (b, a) in within.items():
    out[f"within_before_{lang}"] = b
    out[f"within_after_{lang}"] = a

path = TABLES_DIR / "table7_comet_qn.csv"
out.to_csv(path, index=False)
print(out.T.to_string(header=False))
print(f"\nSaved \u2192 {path}")

## Step 9 — Output Manifest

In [ ]:
print("=== Notebook 06 — output manifest ===")
print("  table7_comet_qn.csv")

## References

**This work.**
Anonymous (2026). *Under review.*

**Information Parity (IP).**
Tsvetkov, A., & Kipnis, A. (2024). Information Parity: Measuring and Predicting the
Multilingual Capabilities of Language Models. *Findings of EMNLP 2024*, pp. 7971–7989.

**Tokenization Parity and tokenizer unfairness.**
Petrov, A., La Malfa, E., Torr, P. H. S., & Bibi, A. (2023). Language Model Tokenizers
Introduce Unfairness Between Languages. *NeurIPS 36*.

**COMET.**
Rei, R., Stewart, C., Farinha, A. C., & Lavie, A. (2020). COMET: A Neural Framework for
MT Evaluation. *EMNLP 2020*, pp. 2685–2702. https://aclanthology.org/2020.emnlp-main.213

**IndicMT Eval dataset.**
Sai B., A., Dixit, T., Nagarajan, V., Kunchukuttan, A., Kumar, P., Khapra, M. M., &
Dabre, R. (2023). IndicMT Eval: A Dataset to Meta-Evaluate Machine Translation Metrics
for Indian Languages. *ACL 2023*, pp. 14210–14228. https://aclanthology.org/2023.acl-long.795

**WMT24 Latin-script controls.**
Kocmi, T., et al. (2024). Findings of the WMT24 General Machine Translation Shared Task.
*Proceedings of WMT 2024*, pp. 1–46. https://aclanthology.org/2024.wmt-1.1